# H₂ Yield Prediction from Dark Fermentation Process Parameters

## Notebook 01: Experimental Data Integration and Quality Audit

66 experimental runs across six F/M ratios (0.5–3.0). Load, audit consistency/outliers, export audited (66) and reference (61) datasets.

## 1. Import Required Libraries

Import pandas, numpy, matplotlib, and seaborn.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## 2. Load the Experimental Dataset

Load raw CSV; verify row/column counts.

In [2]:
df = pd.read_csv(
    r"D:\Data Science Projects\Hydrogen yield predictor\data\raw\data.csv"
)

print("Dataset loaded successfully.")
print(f"Number of observations: {df.shape[0]}")
print(f"Number of variables: {df.shape[1]}")

df.head()

Dataset loaded successfully.
Number of observations: 66
Number of variables: 17


,treatment,substrate_mc(%),substrate_ts(%),substrate_vs(%),substrate_fs(%),inoculum_mc(%),inoculum_ts(%),inoculum_vs(%),incolum_fs(%),scod(mg/L),tcod(mg/L),scod/tcod,trs(mg/L),lignin(%),5-hmf(mg/L),f/m,h2_yield(mL h2/g VS)
0,1,5.210,94.790,89.220,10.780,90.110,9.890,90.110,9.890,6000,11500,0.520,1.440,25.420,42.550,0.500,22.220
1,2,5.640,94.360,87.230,12.770,92.330,7.670,86.400,13.600,6400,12000,0.530,6.820,23.210,689.490,0.500,20.400
2,3,5.610,94.390,88.260,11.740,95.120,4.880,87.260,12.740,5800,8900,0.650,5.740,23.000,425.190,0.500,26.450
3,4,5.260,94.740,78.640,21.360,94.700,5.300,92.330,7.670,5800,9000,0.640,22.160,28.910,189.350,0.500,24.220
4,5,5.940,94.060,78.410,21.590,93.780,6.220,84.770,15.230,6000,11200,0.540,22.200,20.230,0.000,0.500,19.650


## 4. Standardize Variable Names

Strip units and fix incolum_fs → inoculum_fs; values unchanged.

In [3]:
column_mapping = {
    "treatment": "treatment",
    "substrate_mc(%)": "substrate_mc",
    "substrate_ts(%)": "substrate_ts",
    "substrate_vs(%)": "substrate_vs",
    "substrate_fs(%)": "substrate_fs",
    "inoculum_mc(%)": "inoculum_mc",
    "inoculum_ts(%)": "inoculum_ts",
    "inoculum_vs(%)": "inoculum_vs",
    "incolum_fs(%)": "inoculum_fs",
    "scod(mg/L)": "scod",
    "tcod(mg/L)": "tcod",
    "scod/tcod": "scod_tcod_ratio",
    "trs(mg/L)": "trs",
    "lignin(%)": "lignin",
    "5-hmf(mg/L)": "hmf",
    "f/m": "fm_ratio",
    "h2_yield(mL h2/g VS)": "h2_yield"
}

df = df.rename(columns=column_mapping)

print("Column names standardized successfully.\n")

for i, column in enumerate(df.columns, start=1):
    print(f"{i:02d}. {column}")

Column names standardized successfully.

01. treatment
02. substrate_mc
03. substrate_ts
04. substrate_vs
05. substrate_fs
06. inoculum_mc
07. inoculum_ts
08. inoculum_vs
09. inoculum_fs
10. scod
11. tcod
12. scod_tcod_ratio
13. trs
14. lignin
15. hmf
16. fm_ratio
17. h2_yield


## 5. Inspect Dataset Structure and Data Types

Confirm dtypes — all modeling features should be numeric.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   treatment        66 non-null     int64  
 1   substrate_mc     66 non-null     float64
 2   substrate_ts     66 non-null     float64
 3   substrate_vs     66 non-null     float64
 4   substrate_fs     66 non-null     float64
 5   inoculum_mc      66 non-null     float64
 6   inoculum_ts      66 non-null     float64
 7   inoculum_vs      66 non-null     float64
 8   inoculum_fs      66 non-null     float64
 9   scod             66 non-null     int64  
 10  tcod             66 non-null     int64  
 11  scod_tcod_ratio  66 non-null     float64
 12  trs              66 non-null     float64
 13  lignin           66 non-null     float64
 14  hmf              66 non-null     float64
 15  fm_ratio         66 non-null     float64
 16  h2_yield         66 non-null     float64
dtypes: float64(14), in

## 6. Evaluate the Distribution of F/M Ratios

Six F/M levels; count per group (anchor for 1,000 synthetic samples each).

In [5]:
fm_distribution = (
    df["fm_ratio"]
    .value_counts()
    .sort_index()
    .rename_axis("F/M Ratio")
    .reset_index(name="Experimental Observations")
)

fm_distribution

,F/M Ratio,Experimental Observations
0,0.500,11
1,1.000,11
2,1.500,11
3,2.000,11
4,2.500,11
5,3.000,11


### Observation

Balanced: 11 observations per F/M ratio.

## 7. Missing Value and Duplicate Audit

Check missing values and duplicate rows.

In [6]:
missing_values = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage (%)": (df.isnull().sum() / len(df)) * 100
})

missing_values

,Missing Values,Missing Percentage (%)
treatment,0,0.000
substrate_mc,0,0.000
substrate_ts,0,0.000
substrate_vs,0,0.000
substrate_fs,0,0.000
inoculum_mc,0,0.000
inoculum_ts,0,0.000
inoculum_vs,0,0.000
inoculum_fs,0,0.000
scod,0,0.000


In [7]:
total_missing = df.isnull().sum().sum()
duplicate_rows = df.duplicated().sum()

print(f"Total missing values: {total_missing}")
print(f"Duplicate rows: {duplicate_rows}")

Total missing values: 0
Duplicate rows: 0


### Observation

No missing values or duplicates.

## 8. Mathematical and Physical Consistency Audit

Check MC+TS≈100%, VS+FS≈100%, sCOD/TCOD∈[0,1] (±0.5% tolerance).

In [8]:
# Calculate composition consistency

df["substrate_mc_ts_sum"] = (
    df["substrate_mc"] + df["substrate_ts"]
)

df["substrate_vs_fs_sum"] = (
    df["substrate_vs"] + df["substrate_fs"]
)

df["inoculum_mc_ts_sum"] = (
    df["inoculum_mc"] + df["inoculum_ts"]
)

df["inoculum_vs_fs_sum"] = (
    df["inoculum_vs"] + df["inoculum_fs"]
)

df[
    [
        "treatment",
        "substrate_mc_ts_sum",
        "substrate_vs_fs_sum",
        "inoculum_mc_ts_sum",
        "inoculum_vs_fs_sum"
    ]
].head()

,treatment,substrate_mc_ts_sum,substrate_vs_fs_sum,inoculum_mc_ts_sum,inoculum_vs_fs_sum
0,1,100.000,100.000,100.000,100.000
1,2,100.000,100.000,100.000,100.000
2,3,100.000,100.000,100.000,100.000
3,4,100.000,100.000,100.000,100.000
4,5,100.000,100.000,100.000,100.000


### Composition Consistency Criteria

Flag pairs deviating >±0.5 pp from 100%; flags are for review only.

In [9]:
tolerance = 0.5

composition_flags = df[
    (abs(df["substrate_mc_ts_sum"] - 100) > tolerance) |
    (abs(df["substrate_vs_fs_sum"] - 100) > tolerance) |
    (abs(df["inoculum_mc_ts_sum"] - 100) > tolerance) |
    (abs(df["inoculum_vs_fs_sum"] - 100) > tolerance)
]

print(
    f"Number of composition inconsistencies: "
    f"{len(composition_flags)}"
)

composition_flags[
    [
        "treatment",
        "substrate_mc_ts_sum",
        "substrate_vs_fs_sum",
        "inoculum_mc_ts_sum",
        "inoculum_vs_fs_sum"
    ]
]

Number of composition inconsistencies: 0


,treatment,substrate_mc_ts_sum,substrate_vs_fs_sum,inoculum_mc_ts_sum,inoculum_vs_fs_sum


## 9. Audit the sCOD/TCOD Relationship

Verify sCOD≤TCOD, ratio∈[0,1], and reported ratio≈sCOD/TCOD.

In [10]:
df["calculated_scod_tcod_ratio"] = (
    df["scod"] / df["tcod"]
)

df["ratio_difference"] = abs(
    df["scod_tcod_ratio"]
    - df["calculated_scod_tcod_ratio"]
)

ratio_audit = df[
    (df["scod"] > df["tcod"]) |
    (df["scod_tcod_ratio"] < 0) |
    (df["scod_tcod_ratio"] > 1) |
    (df["ratio_difference"] > 0.02)
]

print(
    f"Number of sCOD/TCOD inconsistencies: "
    f"{len(ratio_audit)}"
)

ratio_audit[
    [
        "treatment",
        "scod",
        "tcod",
        "scod_tcod_ratio",
        "calculated_scod_tcod_ratio",
        "ratio_difference"
    ]
]

Number of sCOD/TCOD inconsistencies: 2


,treatment,scod,tcod,scod_tcod_ratio,calculated_scod_tcod_ratio,ratio_difference
29,30,12000,11600,1.030,1.034,0.004
36,37,6600,6400,1.030,1.031,0.001


## 10. Develop a Rule-Based Data Quality Flagging System

Tag alid, composition_inconsistency, scod_exceeds_tcod, or invalid_scod_tcod_ratio; flagged rows retained.

In [11]:
# Initialize all observations as valid

df["data_quality_flag"] = "valid"


# Flag composition inconsistencies

composition_mask = (
    (abs(df["substrate_mc_ts_sum"] - 100) > tolerance) |
    (abs(df["substrate_vs_fs_sum"] - 100) > tolerance) |
    (abs(df["inoculum_mc_ts_sum"] - 100) > tolerance) |
    (abs(df["inoculum_vs_fs_sum"] - 100) > tolerance)
)

df.loc[
    composition_mask,
    "data_quality_flag"
] = "composition_inconsistency"


# Flag observations where sCOD exceeds TCOD

scod_tcod_mask = df["scod"] > df["tcod"]

df.loc[
    scod_tcod_mask,
    "data_quality_flag"
] = "scod_exceeds_tcod"


# Flag invalid reported sCOD/TCOD ratios

invalid_ratio_mask = (
    (df["scod_tcod_ratio"] < 0) |
    (df["scod_tcod_ratio"] > 1)
)

df.loc[
    invalid_ratio_mask,
    "data_quality_flag"
] = "invalid_scod_tcod_ratio"


print("Data quality flagging completed.\n")

df["data_quality_flag"].value_counts()

Data quality flagging completed.



data_quality_flag
valid                      64
invalid_scod_tcod_ratio     2
Name: count, dtype: int64

## 11. Create Multi-Flag Data Quality Records

Record all issues per row (pipe-separated) so overlapping flags are not lost.

In [12]:
def assign_quality_flags(row):
    
    flags = []

    # Composition checks
    
    if abs(row["substrate_mc_ts_sum"] - 100) > tolerance:
        flags.append("substrate_mc_ts_inconsistency")

    if abs(row["substrate_vs_fs_sum"] - 100) > tolerance:
        flags.append("substrate_vs_fs_inconsistency")

    if abs(row["inoculum_mc_ts_sum"] - 100) > tolerance:
        flags.append("inoculum_mc_ts_inconsistency")

    if abs(row["inoculum_vs_fs_sum"] - 100) > tolerance:
        flags.append("inoculum_vs_fs_inconsistency")

    # COD consistency checks
    
    if row["scod"] > row["tcod"]:
        flags.append("scod_exceeds_tcod")

    if not 0 <= row["scod_tcod_ratio"] <= 1:
        flags.append("invalid_scod_tcod_ratio")

    # Return valid if no issue is detected
    
    if len(flags) == 0:
        return "valid"

    return " | ".join(flags)


df["quality_flags"] = df.apply(
    assign_quality_flags,
    axis=1
)

In [13]:
flagged_observations = df[
    df["quality_flags"] != "valid"
]

print(
    f"Total flagged observations: "
    f"{len(flagged_observations)}"
)

flagged_observations[
    [
        "treatment",
        "fm_ratio",
        "scod",
        "tcod",
        "scod_tcod_ratio",
        "quality_flags"
    ]
]

Total flagged observations: 2


,treatment,fm_ratio,scod,tcod,scod_tcod_ratio,quality_flags
29,30,1.500,12000,11600,1.030,scod_exceeds_tcod | invalid_scod_tcod_ratio
36,37,2.000,6600,6400,1.030,scod_exceeds_tcod | invalid_scod_tcod_ratio


### Observation

Two rows: sCOD>TCOD — physical errors, not IQR outliers; kept with flags.

## 12. Group-Wise Statistical Outlier Detection

IQR screening within each F/M group; diagnostic only, not auto-removed.

### 12.1 Select Variables for Statistical Outlier Screening

Exclude IDs, F/M, audit columns, and flags; include H₂ yield.

In [14]:
outlier_columns = [
    "substrate_mc",
    "substrate_ts",
    "substrate_vs",
    "substrate_fs",
    "inoculum_mc",
    "inoculum_ts",
    "inoculum_vs",
    "inoculum_fs",
    "scod",
    "tcod",
    "scod_tcod_ratio",
    "trs",
    "lignin",
    "hmf",
    "h2_yield"
]

print("Variables selected for outlier screening:\n")

for column in outlier_columns:
    print(f"- {column}")

Variables selected for outlier screening:

- substrate_mc
- substrate_ts
- substrate_vs
- substrate_fs
- inoculum_mc
- inoculum_ts
- inoculum_vs
- inoculum_fs
- scod
- tcod
- scod_tcod_ratio
- trs
- lignin
- hmf
- h2_yield


### 12.2 Apply the IQR Method Within Each F/M Group

Apply IQR per F/M group; store flagged variables per row.

In [15]:
def detect_group_outliers(group, columns):
    
    group = group.copy()
    outlier_records = [[] for _ in range(len(group))]
    
    for column in columns:
        
        q1 = group[column].quantile(0.25)
        q3 = group[column].quantile(0.75)
        
        iqr = q3 - q1
        
        lower_bound = q1 - (1.5 * iqr)
        upper_bound = q3 + (1.5 * iqr)
        
        outlier_mask = (
            (group[column] < lower_bound) |
            (group[column] > upper_bound)
        )
        
        for position in np.where(outlier_mask)[0]:
            outlier_records[position].append(column)
    
    group["outlier_variables"] = [
        " | ".join(flags) if flags else "none"
        for flags in outlier_records
    ]
    
    return group

In [16]:
df = (
    df.groupby("fm_ratio", group_keys=False)
    .apply(
        detect_group_outliers,
        columns=outlier_columns
    )
    .reset_index(drop=True)
)

print("Group-wise outlier detection completed.")

Group-wise outlier detection completed.


C:\Users\leovo\AppData\Local\Temp\ipykernel_8604\2022306720.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


## 13. Inspect Detected Statistical Outliers

List rows with IQR flags; separate statistical rarity from physical inconsistency.

In [17]:
statistical_outliers = df[
    df["outlier_variables"] != "none"
]

print(
    f"Total observations with at least one statistical outlier: "
    f"{len(statistical_outliers)}"
)

statistical_outliers[
    [
        "treatment",
        "fm_ratio",
        "outlier_variables",
        "quality_flags"
    ]
]

Total observations with at least one statistical outlier: 36


,treatment,fm_ratio,outlier_variables,quality_flags
6,7,0.500,scod,valid
8,9,0.500,h2_yield,valid
9,10,0.500,scod | tcod,valid
11,12,1.000,substrate_mc | substrate_ts,valid
12,13,1.000,scod | scod_tcod_ratio,valid
14,15,1.000,scod,valid
15,16,1.000,substrate_vs | substrate_fs,valid
16,17,1.000,substrate_vs | substrate_fs,valid
22,23,1.500,substrate_mc | substrate_ts,valid
23,24,1.500,substrate_mc | substrate_ts,valid


In [18]:
outlier_summary = (
    statistical_outliers["outlier_variables"]
    .str.split(" | ", regex=False)
    .explode()
    .value_counts()
    .rename_axis("Variable")
    .reset_index(name="Outlier Count")
)

outlier_summary

,Variable,Outlier Count
0,scod,10
1,substrate_vs,6
2,substrate_fs,6
3,hmf,6
4,substrate_mc,5
5,substrate_ts,5
6,tcod,5
7,trs,5
8,scod_tcod_ratio,5
9,h2_yield,4


## 14. Interpretation of Group-Wise IQR Outlier Screening

36 rows flagged, but with n=11 per F/M group IQR bounds are tight — **IQR flags ≠ bad data.**

In [19]:
known_anomaly_treatments = [30, 31, 37, 46, 62]

df["high_confidence_anomaly"] = df["treatment"].isin(
    known_anomaly_treatments
)

df[
    df["high_confidence_anomaly"]
][
    [
        "treatment",
        "fm_ratio",
        "substrate_vs",
        "inoculum_vs",
        "scod",
        "tcod",
        "scod_tcod_ratio",
        "outlier_variables",
        "quality_flags"
    ]
]

,treatment,fm_ratio,substrate_vs,inoculum_vs,scod,tcod,scod_tcod_ratio,outlier_variables,quality_flags
29,30,1.500,87.000,91.240,12000,11600,1.030,scod | scod_tcod_ratio,scod_exceeds_tcod | invalid_scod_tcod_ratio
30,31,1.500,56.000,89.260,5700,9500,0.600,substrate_vs | substrate_fs,valid
36,37,2.000,89.990,89.680,6600,6400,1.030,tcod | scod_tcod_ratio,scod_exceeds_tcod | invalid_scod_tcod_ratio
45,46,2.500,90.940,86.660,6700,125000,0.050,tcod | scod_tcod_ratio,valid
61,62,3.000,88.360,8.930,6400,14000,0.460,inoculum_vs | inoculum_fs,valid


### Observation

Five high-confidence anomalies (treatments 30, 31, 37, 46, 62) confirmed by independent checks.

## 16. Create a Quality-Controlled Reference Dataset

Keep all 66 raw rows; exclude five high-confidence anomalies from reference set (61 rows).

In [20]:
reference_df = df[
    ~df["high_confidence_anomaly"]
].copy()

print("Original experimental observations:", len(df))
print("High-confidence anomalies:", df["high_confidence_anomaly"].sum())
print("Reference observations:", len(reference_df))

Original experimental observations: 66
High-confidence anomalies: 5
Reference observations: 61


In [21]:
reference_distribution = (
    reference_df["fm_ratio"]
    .value_counts()
    .sort_index()
    .rename_axis("F/M Ratio")
    .reset_index(name="Reference Observations")
)

reference_distribution

,F/M Ratio,Reference Observations
0,0.500,11
1,1.000,11
2,1.500,9
3,2.000,10
4,2.500,10
5,3.000,10


### Observation

Reference: 61 rows (5 anomalies removed); raw 66-row file unchanged.

## 17. Save the Audited and Quality-Controlled Datasets

Save audited (66) with all flags; reference (61) excluding high-confidence anomalies.

In [22]:
output_path = (
    r"D:\Data Science Projects\Hydrogen yield predictor\data"
)

audited_file = (
    output_path + r"\processed\audited_experimental_data.csv"
)

reference_file = (
    output_path + r"\processed\reference_experimental_data.csv"
)

df.to_csv(
    audited_file,
    index=False
)

reference_df.to_csv(
    reference_file,
    index=False
)

print("Datasets saved successfully.")
print("\nAudited dataset:")
print(audited_file)

print("\nReference dataset:")
print(reference_file)

Datasets saved successfully.

Audited dataset:
D:\Data Science Projects\Hydrogen yield predictor\data\audited_experimental_data.csv

Reference dataset:
D:\Data Science Projects\Hydrogen yield predictor\data\reference_experimental_data.csv


## 18. Verify the Saved Datasets

Reload both CSVs; confirm shape and F/M counts.

In [23]:
audited_check = pd.read_csv(audited_file)
reference_check = pd.read_csv(reference_file)

print("Audited dataset shape:")
print(audited_check.shape)

print("\nReference dataset shape:")
print(reference_check.shape)

print("\nF/M distribution in reference dataset:")
print(
    reference_check["fm_ratio"]
    .value_counts()
    .sort_index()
)

Audited dataset shape:
(66, 27)

Reference dataset shape:
(61, 27)

F/M distribution in reference dataset:
fm_ratio
0.500    11
1.000    11
1.500     9
2.000    10
2.500    10
3.000    10
Name: count, dtype: int64


## 19. Notebook Summary

66 runs audited → composition/COD checks, IQR screening (not auto-removal), 5 anomalies excluded → audited (66) + reference (61) saved.